# Demo 2 & 3 — The pieces are sub-word tokens
<a id="top"></a>

```yaml
title: Demo 2 & 3 — The pieces are sub-word tokens
keywords: tokens, sub-word, bpe, tiktoken, tokenization, chars per token, json, non-latin
estimated duration: 14
```

> **Lesson L01**, the second and third demos in the chain. Written lecture
> [L0102_lecture.md](L0102_lecture.md) §2–3.
>
> `tiktoken` is offline and lets you **see** sub-word boundaries. (For real Claude cost/window
> math, use the model's own token count — the *boundaries* here are for building intuition.)

**Contents**

1. [The next piece is a token](#1-the-next-piece-is-a-token)
2. [Why sub-word, not words or characters](#2-why-sub-word-not-words-or-characters)
3. [What tokenization does to real strings](#3-what-tokenization-does-to-real-strings)

In [1]:
from __future__ import annotations

import tiktoken

ENC = tiktoken.get_encoding("cl100k_base")  # a common BPE tokenizer


def count_tokens(text: str) -> int:
    return len(ENC.encode(text))


def token_pieces(text: str) -> list[str]:
    """Decode each token separately so the sub-word boundaries are visible."""
    return [ENC.decode([tid]) for tid in ENC.encode(text)]


def show(text: str) -> None:
    print(f"{count_tokens(text):>3} tok | {'|'.join(token_pieces(text))}")

## 1. The next piece is a token

The unit the model predicts is a **token** — often a whole common word, often a fragment.
A common word is usually one token; a rare or invented word is spelled out in several.

In [2]:
for word in ["running", "antidisestablishmentarianism", "flibbertigibbeting"]:
    show(word)

  1 tok | running
  6 tok | ant|idis|establish|ment|arian|ism
  7 tok | fl|ib|bert|ig|ib|b|eting


## 2. Why sub-word, not words or characters

- **Not whole words:** an *unbounded* vocabulary — every name, typo, and new word would need
  its own entry, and you'd still miss tomorrow's words.
- **Not single characters:** sequences get very long and each step carries little signal.
- **Sub-word is the compromise:** a *fixed* vocabulary of common chunks that **compose** to
  spell anything. Common strings get a short encoding; rare ones pay in pieces.

A token is really just an integer ID — the readable pieces above are a decoding for you:

In [3]:
print(ENC.encode("running"))  # e.g. [running] -> a couple of integer IDs
print(ENC.encode("flibbertigibbeting"))

[28272]
[1517, 581, 9339, 343, 581, 65, 11880]


## 3. What tokenization does to real strings

Because the vocabulary is *learned* from training data, anything **rare** in that data gets
spelled out in many pieces — proper names, code, JSON, non-Latin scripts. These are exactly
the inputs agents marinate in, which is why eyeball estimates fail where it costs you.

In [4]:
SAMPLES = {
    "english": "The quick brown fox jumps over the lazy dog.",
    "names": "Nnamdi Azikiwe deployed Kubernetes to Reykjavik",
    "code": "def f(x): return x ** 2",
    "json": '{"user_id": 12345, "events": ["click", "scroll"]}',
    "japanese": "東京は日本の首都です",
}
for label, text in SAMPLES.items():
    print(f"\n[{label}]")
    show(text)


[english]
 10 tok | The| quick| brown| fox| jumps| over| the| lazy| dog|.

[names]
 13 tok | N|nam|di| Az|iki|we| deployed| Kubernetes| to| Rey|k|jav|ik

[code]
  9 tok | def| f|(x|):| return| x| **| |2

[json]
 17 tok | {"|user|_id|":| |123|45|,| "|events|":| ["|click|",| "|scroll|"]}

[japanese]
 10 tok | �|�|京|は|日|本|の|首|都|です


In [5]:
# The "~4 chars per token" rule holds for English and breaks everywhere agents live.
for label, text in SAMPLES.items():
    n = count_tokens(text)
    print(f"{label:9s} {len(text):>3} chars / {n:>2} tok = {len(text) / n:4.1f} chars/token")

english    44 chars / 10 tok =  4.4 chars/token
names      47 chars / 13 tok =  3.6 chars/token
code       23 chars /  9 tok =  2.6 chars/token
json       49 chars / 17 tok =  2.9 chars/token
japanese   10 chars / 10 tok =  1.0 chars/token


English lands near ~4 chars/token; names fragment; JSON is much denser (every brace/quote is a
token); non-Latin explodes. **Every cost and window number later in the lesson is downstream of
this count** — get tokens wrong and every estimate is wrong by the same factor.

[↑ Back to top](#top)